# Gene Ontology Reproducibility Analysis

## Purpose: 

{TODO}

In [1]:
import pandas as pd
pd.set_option('display.max_rows', 1000000)
pd.set_option('display.max_columns', 1000000)
pd.set_option('display.max_colwidth', 10000000)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import pickle, glob

import seaborn as sns
import matplotlib.pyplot as plt

## Load Data

In [2]:
meta_GO_df = pd.read_csv(
    "../../../../outputs/gene_ontology_analysis/differential_expression_networks/all_GO_analyses_p_uncorrected_0.05.tsv.gz", 
    sep="\t", 
    compression="gzip"
)

baseline_networks_GO = pd.read_csv(
    "../../../../outputs/gene_ontology_analysis/differential_expression_networks/baseline_networks_GO_p_uncorrected_0.05.tsv.gz", 
    sep="\t", 
    compression='gzip'
)

In [3]:
meta_louvain_clustering = pickle.load(
    open("../../../../outputs/louvain_clustering/differential_expression_networks/baseline_network_louvain_clustering.pkl", 'rb')
    
)

baseline_louvain_clustering =pickle.load(
    open("../../../../outputs/louvain_clustering/differential_expression_networks/baseline_network_louvain_clustering.pkl", 'rb')

)

In [4]:
# subset to biological processes 
meta_GO_df = meta_GO_df[meta_GO_df["NS"]=="BP"]
baseline_networks_GO = baseline_networks_GO[baseline_networks_GO["NS"]=="BP"]

meta_GO_df.index.size
meta_GO_df.head()

baseline_networks_GO.index.size 
baseline_networks_GO.head()

22664465

,# GO,NS,enrichment,name,ratio_in_study,ratio_in_pop,p_uncorrected,depth,study_count,p_fdr_bh,study_items,Parameter_Combination,Iteration,Cluster
0,GO:0014860,BP,e,neurotransmitter secretion involved in regulation of skeletal muscle contraction,1/19,1/21965,0.000865,9,1,1.0,nr4a1,upstream_2500_score_900,0,cluster_6
1,GO:0018900,BP,e,dichloromethane metabolic process,1/19,1/21965,0.000865,5,1,1.0,gstt1,upstream_2500_score_900,0,cluster_6
2,GO:1903954,BP,e,positive regulation of voltage-gated potassium channel activity involved in atrial cardiac muscle cell action potential repolarization,1/19,1/21965,0.000865,12,1,1.0,rnf207,upstream_2500_score_900,0,cluster_6
3,GO:1903762,BP,e,positive regulation of voltage-gated potassium channel activity involved in ventricular cardiac muscle cell action potential repolarization,1/19,1/21965,0.000865,12,1,1.0,rnf207,upstream_2500_score_900,0,cluster_6
4,GO:1900542,BP,e,regulation of purine nucleotide metabolic process,1/19,1/21965,0.000865,8,1,1.0,lacc1,upstream_2500_score_900,0,cluster_6


22272

,# GO,NS,enrichment,name,ratio_in_study,ratio_in_pop,p_uncorrected,depth,study_count,p_fdr_bh,study_items,Parameter_Combination,Iteration,Cluster
0,GO:1905703,BP,e,negative regulation of inhibitory synapse assembly,1/13,1/21965,0.000592,8,1,1.0,cbln1,upstream_2500_score_900,1000,cluster_1
1,GO:0070194,BP,e,synaptonemal complex disassembly,1/13,2/21965,0.001183,7,1,1.0,plk1,upstream_2500_score_900,1000,cluster_1
2,GO:1904776,BP,e,regulation of protein localization to cell cortex,1/13,2/21965,0.001183,7,1,1.0,plk1,upstream_2500_score_900,1000,cluster_1
3,GO:2001213,BP,e,negative regulation of vasculogenesis,1/13,2/21965,0.001183,6,1,1.0,xdh,upstream_2500_score_900,1000,cluster_1
4,GO:0010044,BP,e,response to aluminum ion,1/13,2/21965,0.001183,5,1,1.0,xdh,upstream_2500_score_900,1000,cluster_1


## For each parameter combination, how many unique terms are found in each iteration that are not in the baseline network?

In [5]:
# use string manipulation to pull out parameter combinations

parameter_combinations = [file.split("/")[-1].split(".")[0] for file in sorted(glob.glob("/home/jve4pt/RBFOX2-Mouse-Embryonic-Heart-Analysis/outputs/pickled_networks/differential_expression_tf_networks/*", recursive=True))]

parameter_combinations

['upstream_2500_score_900',
 'upstream_2500_score_950',
 'upstream_2500_score_990',
 'upstream_5000_score_900',
 'upstream_5000_score_950',
 'upstream_5000_score_990',
 'upstream_750_score_900',
 'upstream_750_score_950',
 'upstream_750_score_990']

In [ ]:

# create dataframe for downstream plotting 
# to see if the distribution of number of unique terms not found in baseline network
not_in_baseline_network_df = pd.DataFrame()

for parameter_combination in parameter_combinations: 
    
    # get list of baseline network GO terms
    baseline_tmp = baseline_networks_GO[baseline_networks_GO["Parameter_Combination"] == parameter_combination]["# GO"].unique().tolist()
    
    # for each iteration 
    for iteration in meta_GO_df["Iteration"].unique().tolist(): 
        
        # subset to parameter combination and iteration 
        # pull out unique GO IDs for that iteration and parameter combination
        iteration_tmp = meta_GO_df[(meta_GO_df["Parameter_Combination"]==parameter_combination) & (meta_GO_df["Iteration"]==iteration)]["# GO"].unique().tolist()
        
        
        # concatenate original dataframe with this one which includes 
        # parameter combination and number of unique terms not found in baseline network
        not_in_baseline_network_df = pd.concat([not_in_baseline_network_df, pd.DataFrame([[parameter_combination, len(set(iteration_tmp) - set(baseline_tmp))]])])

In [ ]:
not_in_baseline_network_df.head()

In [ ]:
plt.figure(dpi=150)

sns.boxplot(data=not_in_baseline_network_df, x=0, y=1)

plt.xticks(rotation=90)
plt.xlabel("Parameter Combination")
plt.ylabel("Number of GO Terms Not Seen in Baseline Network")

## For any term found across any of the 1000 iterations, how many iterations is it found in total? 

Number of Unique `GO` terms

In [ ]:
meta_GO_df["# GO"].unique().size

In [ ]:
term_reproducibility = pd.DataFrame()

for parameter_combination in parameter_combinations: 
        
    parameter_combo_df = meta_GO_df[meta_GO_df["Parameter_Combination"]==parameter_combination]
    
    for GO_term in parameter_combo_df["# GO"].unique().tolist(): 
        
        tmp_num = parameter_combo_df[parameter_combo_df["# GO"]==GO_term]["Iteration"].unique().size
        
        term_reproducibility = pd.concat(
            [
                term_reproducibility, 
                pd.DataFrame([[parameter_combination, GO_term, tmp_num]])
            ]
        )
        

In [ ]:
term_reproducibility.columns = ["Parameter Combination", "GO Term", "Number Iterations Found"]

term_reproducibility.head()

In [ ]:
term_reproducibility["Number Iterations Found"].describe()

In [ ]:
plt.figure(dpi=200)
sns.boxplot(data = term_reproducibility, x="Parameter Combination", y="Number Iterations Found")
plt.xticks(rotation=90)

plt.figure(dpi=150)
sns.stripplot(data=term_reproducibility, x="Parameter Combination", y="Number Iterations Found", size=2)
plt.xticks(rotation=90)

## For each GO term in a parameter combination's baseline network, what percentage of iterations include that term?

In [ ]:
term_reproducibility = pd.DataFrame()

for parameter_combination in parameter_combinations: 
        
    parameter_combo_df = meta_GO_df[meta_GO_df["Parameter_Combination"]==parameter_combination]
    
    for GO_term in baseline_networks_GO[baseline_networks_GO["Parameter_Combination"]==parameter_combination]["# GO"].unique().tolist(): 
        
        tmp_num = (parameter_combo_df[parameter_combo_df["# GO"]==GO_term]["Iteration"].unique().size)/1000
        
        term_reproducibility = pd.concat(
            [
                term_reproducibility, 
                pd.DataFrame([[parameter_combination, GO_term, tmp_num]])
            ]
        )


term_reproducibility.columns = ["Parameter Combination", "GO Term", "Number Iterations Found"]

term_reproducibility.head()
        
term_reproducibility["Number Iterations Found"].describe()

plt.figure(dpi=150)
sns.boxplot(data = term_reproducibility, x="Parameter Combination", y="Number Iterations Found")
plt.xticks(rotation=90)

plt.figure(dpi=150)
sns.stripplot(data=term_reproducibility, x="Parameter Combination", y="Number Iterations Found", size=5)
plt.xticks(rotation=90)

## For Terms that are Found in Multiple Iterations, What is the Jaccard Score Across Each Term?

In [ ]:
jaccard_similarities = pd.DataFrame()

for parameter_combination in parameter_combinations: 
        
    parameter_combo_df = meta_GO_df[meta_GO_df["Parameter_Combination"]==parameter_combination]
    
    for GO_term in parameter_combo_df["# GO"].unique().tolist(): 
        
        parameter_combo_df[parameter_combo_df["# GO"]==GO_term].head()

        

In [ ]:
meta_GO_df["enrichment"].value_counts()